# Analiza sieciowa swipe'ów: (b) Wzajemność ponad propensje

**Pytanie badawcze:** Czy liczba matchy w parach z decyzją obu stron przewyższa
liczbę oczekiwaną przy niezależności decyzji ($\sum \hat{p}_{AB} \cdot \hat{p}_{BA}$)?

**Uniwersum analizy:** 7 715 nieuporządkowanych par {A, B} z decyzją obu stron
(482 zrekonstruowane matche).
**Cross-fitting:** Prawdopodobieństwa $\hat{p}_{AB}$ i $\hat{p}_{BA}$ pochodzą z modeli
gdzie krawędzie pary znajdowały się w zbiorze testowym (brak wycieku etykiety zwrotnej).

> [!NOTE]
> Zgodnie z ograniczeniem G1: decyzja strony B jest obserwowana tylko wtedy, gdy feed
> pokazał profil A użytkownikowi B. Wnioski dotyczą zachowania uwarunkowanego ekspozycją.

In [ ]:
import json

import matplotlib.pyplot as plt
from PIL import Image

from rentfriends.graph.data import build_funnel, load_raw, verify_h1_sha256
from rentfriends.graph.reciprocity import build_pair_universe

verify_h1_sha256()
raw = load_raw(verify_sha=True)
df, funnel = build_funnel(raw.h1)
pair_df = build_pair_universe(df)
n_obs = pair_df["match_observed"].sum()
print(f"Liczba par z obustronną decyzją: {len(pair_df):,}")
print(f"Obserwowane matche: {n_obs:,}")

## 1. Obserwowane vs oczekiwane matche

Oczekiwana liczba matchy liczona jest z sumy iloczynów cross-fitted prawdopodobieństw
$\hat{p}_{AB} \cdot \hat{p}_{BA}$ z modelu M1.
Przedziały ufności 95% estymowane za pomocą **node bootstrapu**.

In [ ]:
with open("../reports/graph/numbers.json") as f:
    numbers = json.load(f)

res_b = numbers["results_b"]
obs = res_b["observed_matches"]
exp = res_b["expected_matches"]
ratio = res_b["ratio_obs_exp"]
ci_lo = res_b["bootstrap_ci_lo"]
ci_hi = res_b["bootstrap_ci_hi"]

print(f"Obserwowane matche: {obs:,}")
print(f"Oczekiwane matche (M1 null): {exp:.1f}")
print(f"Wskaźnik wzajemności (obs / exp): {ratio:.3f}")
print(f"95% Node Bootstrap CI: [{ci_lo:.3f}, {ci_hi:.3f}]")

if ci_lo > 1.0:
    print("Wniosek: Istotna nadwyżka matchy ponad niezależność (+41%).")
else:
    print("Wniosek: Brak dowodu na wzajemność ponad propensje.")

## 2. Rozkład wg koszyków oczekiwanej szansy matcha

Wykres przedstawia obserwowane vs oczekiwane matche w 5 koszykach szansy.

In [ ]:
img_recip = Image.open("../reports/graph/fig_reciprocity_observed_vs_expected.png")
plt.figure(figsize=(10, 5))
plt.imshow(img_recip)
plt.axis("off")
plt.title("Obserwowane vs oczekiwane matche wg koszyków", fontsize=12)
plt.show()

## 3. Ograniczenia i wrażliwość (G1, G15)

- **Wrażliwość G15**: w bazie znajduje się 246 matchy z jednym swipe'em (wykluczone).
- **Ekspozycja (G1)**: Użytkownik B widział profil A tylko jeśli algorytm mu go podał.
Wnioski dotyczą zachowania uwarunkowanego ekspozycją.